# Evaluate BLEU / WER / CER

Install dependencies from the repo root (`pip install -r requirements.txt`). Optional: `pip install -e .` so imports work regardless of the notebook working directory.

In [ ]:
%pip install -q sacrebleu torchmetrics

In [ ]:
from pathlib import Path
import sys

_root = Path.cwd().resolve()
if not (_root / "nmt").is_dir() and (_root.parent / "nmt").is_dir():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import os
os.chdir(_root)

import torch
import sacrebleu
from tqdm.auto import tqdm
import torchmetrics
from nmt.config import get_config, get_weights_path
from nmt.checkpoint import load_training_checkpoint
from nmt.train import get_model, get_dataset, greedy_decode

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = get_config()
train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_dataset(config)
model = get_model(config, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)
model_filename = get_weights_path(config, str(config["checkpoint_epoch"]))
state = load_training_checkpoint(model_filename, map_location=device)
model.load_state_dict(state["model_state_dict"])
model.eval()

In [ ]:
def compute_bleu(model, dataloader, tokenizer_src, tokenizer_tgt, config, device, num_batches=100):
    references = []
    hypotheses = []
    wer = torchmetrics.text.WordErrorRate()
    cer = torchmetrics.text.CharErrorRate()

    with torch.no_grad():
        for i, batch in enumerate(tqdm(dataloader, desc="Computing metrics", total=num_batches)):
            if i >= num_batches:
                break
            encoder_input = batch["encoder_input"].to(device)
            encoder_mask = batch["encoder_mask"].to(device)
            tgt_text = batch["tgt_text"][0]
            model_output = greedy_decode(
                model,
                encoder_input,
                encoder_mask,
                tokenizer_src,
                tokenizer_tgt,
                config["seq_len"],
                device,
            )
            pred = tokenizer_tgt.decode(model_output.cpu().numpy())
            references.append([tgt_text])
            hypotheses.append(pred)
            wer.update(pred, tgt_text)
            cer.update(pred, tgt_text)

    bleu = sacrebleu.corpus_bleu(hypotheses, list(zip(*references)))
    print(f"BLEU score: {bleu.score:.2f}")
    print(f"WER: {wer.compute():.4f}")
    print(f"CER: {cer.compute():.4f}")
    return bleu, wer.compute(), cer.compute()

In [ ]:
compute_bleu(model, val_dataloader, tokenizer_src, tokenizer_tgt, config, device, num_batches=100)

In [ ]:
# TensorBoard (log dir is under the repo root: runs/)
%load_ext tensorboard
%tensorboard --logdir=runs